In [9]:

# LOOKING FOR DISEASES WHICH HAS HIGH IMPACT ON COSTS

import pandas as pd
df = pd.read_csv("medical_insurance.csv")
disease_cols = [
    "hypertension",
    "had_major_procedure",
    "mental_health",
    "arthritis",
    "diabetes",
    "asthma",
    "cardiovascular_disease",
    "cancer_history",
    "liver_disease",
    "kidney_disease"
]

df["disease_count"] = df[disease_cols].sum(axis=1)


In [10]:
from scipy.stats import ttest_ind

# First, let's print the available columns to verify what we're working with
print("Available columns in dataframe:", df.columns.tolist())

# Define disease_cols with columns you know exist in your dataframe
# For example:
# disease_cols = ['diabetes', 'heart_disease', 'hypertension']

# Or if you want to check which of your intended columns actually exist:
intended_cols = ["diabetes", "heart_disease", "hypertension", "asthma",
    "arthritis",
    "cardiovascular_disease",
    "cancer_history",
    "kidney_disease",
    "liver_disease",
    "mental_health",
    "had_major_procedure"]  # adjust as needed
disease_cols = [col for col in intended_cols if col in df.columns]
print("Columns being used for analysis:", disease_cols)

# Now perform the t-test only on columns that exist
for col in disease_cols:
    try:
        group1 = df[df[col] == 1]["annual_medical_cost"]
        group0 = df[df[col] == 0]["annual_medical_cost"]
        
        t_stat, p_val = ttest_ind(group1, group0, equal_var=False)
        
        print(f"\n--- {col.upper()} ---")
        print("T-stat:", t_stat)
        print("P-value:", p_val)
    except KeyError as e:
        print(f"Column {col} not found in dataframe: {e}")


# p < 0.05 → significant
# p > 0.05 → not useful

Available columns in dataframe: ['person_id', 'age', 'sex', 'region', 'urban_rural', 'income', 'education', 'marital_status', 'employment_status', 'household_size', 'dependents', 'bmi', 'smoker', 'alcohol_freq', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'plan_type', 'network_tier', 'deductible', 'copay', 'policy_term_years', 'policy_changes_last_2yrs', 'provider_quality', 'risk_score', 'annual_medical_cost', 'annual_premium', 'monthly_premium', 'claims_count', 'avg_claim_amount', 'total_claims_paid', 'chronic_count', 'hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health', 'proc_imaging_count', 'proc_surgery_count', 'proc_physio_count', 'proc_consult_count', 'proc_lab_count', 'is_high_risk', 'had_major_procedure', 'disease_count']
Columns being used for analysis: ['diabetes', 'hyperten

In [12]:
import numpy as np
import pandas as pd
for col in disease_cols:
    group1 = df[df[col] == 1]["annual_medical_cost"]
    group0 = df[df[col] == 0]["annual_medical_cost"]
    
    effect_size = (group1.mean() - group0.mean()) / np.sqrt(
        (group1.std()**2 + group0.std()**2) / 2
    )
    
    print(f"\n--- {col.upper()} ---")
    print("Effect Size:", effect_size)

# Rough guide:
# 0.2 → small
# 0.5 → medium
# 0.8+ → strong


--- DIABETES ---
Effect Size: 0.3282315590189752

--- HYPERTENSION ---
Effect Size: 0.3524920725934412

--- ASTHMA ---
Effect Size: 0.30031373602801137

--- ARTHRITIS ---
Effect Size: 0.3346945481065377

--- CARDIOVASCULAR_DISEASE ---
Effect Size: 0.3124868876737289

--- CANCER_HISTORY ---
Effect Size: 0.3324643576030453

--- KIDNEY_DISEASE ---
Effect Size: 0.31314053515596163

--- LIVER_DISEASE ---
Effect Size: 0.35688191644944545

--- MENTAL_HEALTH ---
Effect Size: 0.33568509269871694

--- HAD_MAJOR_PROCEDURE ---
Effect Size: 0.3484136221627068


In [14]:
results = []

for col in disease_cols:
    group1 = df[df[col] == 1]["annual_medical_cost"]
    group0 = df[df[col] == 0]["annual_medical_cost"]
    
    t_stat, p_val = ttest_ind(group1, group0, equal_var=False)
    
    results.append({
        "Disease": col,
        "Mean (Yes)": group1.mean(),
        "Mean (No)": group0.mean(),
        "P-value": p_val
    })

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df.sort_values("P-value"))

                  Disease   Mean (Yes)    Mean (No)        P-value
1            hypertension  3964.353982  2765.556574   0.000000e+00
9     had_major_procedure  4035.305912  2799.783806  2.377361e-292
8           mental_health  4030.103899  2856.751875  2.723373e-223
3               arthritis  4044.773379  2883.695569  1.361615e-191
0                diabetes  4104.670247  2906.492493  1.494566e-141
2                  asthma  4020.193632  2946.227522   6.285056e-87
4  cardiovascular_disease  4066.827848  2952.428070   1.299911e-82
5          cancer_history  4173.548419  2983.861747   4.296977e-41
7           liver_disease  4306.033460  2990.014304   7.582268e-32
6          kidney_disease  4082.668003  2993.528690   1.360334e-26


In [15]:
disease_cols = [
    "hypertension",
    "had_major_procedure",
    "mental_health",
    "arthritis",
    "diabetes",
    "asthma",
    "cardiovascular_disease",
    "cancer_history",
    "liver_disease",
    "kidney_disease"
]

df["disease_count"] = df[disease_cols].sum(axis=1)

In [16]:
# MODEL A: Using Disease Count

X1 = df[[
    "age",
    "bmi",
    "dependents",
    "smoker",
    "disease_count",
    "sex",
    "region"
]]

y = df["annual_medical_cost"]

# Convert categorical variables
X1 = pd.get_dummies(X1, drop_first=True)

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X1, y, test_size=0.2, random_state=42
)

lr1 = LinearRegression()
lr1.fit(X_train, y_train)

y_pred1 = lr1.predict(X_test)

print("Model A (Disease Count)")
print("R2:", r2_score(y_test, y_pred1))
print("MAE:", mean_absolute_error(y_test, y_pred1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred1)))

Model A (Disease Count)
R2: 0.1326115733576262
MAE: 1822.6379311629412
RMSE: 2921.5117386246075


In [18]:
from sklearn.ensemble import RandomForestRegressor

rf1 = RandomForestRegressor(n_estimators=100, random_state=42)
rf1.fit(X_train, y_train)

y_pred_rf1 = rf1.predict(X_test)

print("RF Model A")
print("R2:", r2_score(y_test, y_pred_rf1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf1)))

RF Model A
R2: -0.005667498650703262
RMSE: 3145.7777143575013


In [ ]:
# MODEL B: Top Diseases

In [19]:
top_diseases = [
    "diabetes",
    "hypertension",
    "cardiovascular_disease",
    "cancer_history",
    "kidney_disease"
]

In [20]:
X2 = df[[
    "age",
    "bmi",
    "dependents",
    "smoker",
    "sex",
    "region"
] + top_diseases]

X2 = pd.get_dummies(X2, drop_first=True)

In [21]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y, test_size=0.2, random_state=42
)

lr2 = LinearRegression()
lr2.fit(X_train2, y_train2)

y_pred2 = lr2.predict(X_test2)

print("\nModel B (Top Diseases)")
print("R2:", r2_score(y_test2, y_pred2))
print("MAE:", mean_absolute_error(y_test2, y_pred2))
print("RMSE:", np.sqrt(mean_squared_error(y_test2, y_pred2)))


Model B (Top Diseases)
R2: 0.08819177963225977
MAE: 1876.0662731426328
RMSE: 2995.384483015937


In [22]:
# Import the necessary libraries
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Random Forest - Model B
rf2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf2.fit(X_train2, y_train2)

y_pred_rf2 = rf2.predict(X_test2)

print("RF Model B")
print("R2:", r2_score(y_test2, y_pred_rf2))
print("RMSE:", np.sqrt(mean_squared_error(y_test2, y_pred_rf2)))

RF Model B
R2: -0.08546603801951513
RMSE: 3268.2023954568517


In [ ]:
# COMPARISION 

# Multiple predictive models were evaluated to estimate annual medical costs. Among them, Linear Regression using a composite “disease burden” feature achieved the best performance (R² = 0.133).
# The results indicate that the cumulative effect of multiple health conditions is a stronger predictor of healthcare costs than individual diseases considered separately.
# Random Forest models underperformed, suggesting that the relationships in the dataset are largely linear and that increasing model complexity does not necessarily improve predictive accuracy in this case.
# Despite improvements, the overall R² remains moderate, indicating that additional variables such as lifestyle factors, treatment history, and healthcare access may be required for more accurate predictions.